## **Airports**

In [240]:
from selenium import webdriver
from bs4 import BeautifulSoup
import time

driver = webdriver.Chrome()
driver.get("https://www.ind.com/flights")

time.sleep(5) # Waiting unitl the data is loaded

soup = BeautifulSoup(driver.page_source, "html.parser")

value = []
table = soup.select_one("table.flight-destinations__table")
for row in table.select("tbody"):
    cols = row.find_all("td")
    row_values = []
    for col in cols:
        text = col.text.strip()        
        combined = text
        row_values.append(combined)

    value.append(row_values)

value = value[0]
driver.quit()

In [241]:
import pandas as pd
df = pd.DataFrame(value, columns=['AirportName'])
df['AirportCode'] = df['AirportName'].apply(lambda x: x.split('(')[-1].split(')')[0])
df['AirportName'] = df['AirportName'].apply(lambda x: x.split('*')[0].split('(')[0])
df.loc[len(df)] = ['Indianapolis', 'IND']
df_airports = df
df_airports

,AirportName,AirportCode
0,Atlanta,ATL
1,Austin-Bergstrom,AUS
2,Baltimore,BWI
3,Boston,BOS
4,Cancun,CUN
5,Charleston,CHS
6,Charlotte,CLT
7,Chicago-Midway,MDW
8,Chicago O'Hare,ORD
9,Dallas-Fort Worth,DFW


In [242]:
import pymysql

conn = pymysql.connect(
    host='localhost',
    user='root',
    password='root',
    db='flight_data',
    port=8889,
    charset='utf8mb4'
)
cursor = conn.cursor()

sql = """
INSERT INTO Airports (AirportCode, AirportName)
VALUES (%s, %s)
"""
    
values = df_airports[['AirportCode', 'AirportName']].values.tolist()

cursor.executemany(sql, values)
conn.commit()
conn.close()

## **Airlines**

In [243]:
from selenium import webdriver
from bs4 import BeautifulSoup
import time

driver = webdriver.Chrome()
driver.get("https://www.ind.com/flights")

time.sleep(5) # Waiting unitl the data is loaded

soup = BeautifulSoup(driver.page_source, "html.parser")

value = []
table = soup.select_one("table.flight-airlines__table")
for row in table.select("tbody"):
    cols = row.find_all("td")
    row_values = []
    for col in cols:
        text = col.text.strip()        
        combined = text
        row_values.append(combined)

    value.append(row_values)

value = value[0]
value = value[0::3]

driver.quit()

In [244]:
df = pd.DataFrame(value, columns=['AirlineName'])
AirlineCode = pd.Series(['AC', 'AS', 'G4', 'AA', 'DL', 'F9', 'WN', 'NK','SY', 'UA'])
df['AirlineCode'] = AirlineCode
df_airlines = df
df_airlines

,AirlineName,AirlineCode
0,Air Canada,AC
1,Alaska Air,AS
2,Allegiant,G4
3,American,AA
4,Delta,DL
5,Frontier,F9
6,Southwest,WN
7,Spirit,NK
8,Sun Country,SY
9,United,UA


In [245]:
import pymysql

conn = pymysql.connect(
    host='localhost',
    user='root',
    password='root',
    db='flight_data',
    port=8889,
    charset='utf8mb4'
)
cursor = conn.cursor()

sql = """
INSERT INTO Airlines (AirlineCode, AirlineName)
VALUES (%s, %s)
"""
    
values = df_airlines[['AirlineCode', 'AirlineName']].values.tolist()

cursor.executemany(sql, values)
conn.commit()
conn.close()

## **Remarks**

In [246]:
import pymysql

conn = pymysql.connect(
    host='localhost',
    user='root',
    password='root',
    db='flight_data',
    port=8889,
    charset='utf8mb4'
)
cursor = conn.cursor()

sql = "INSERT INTO Remarks (RemarkCode, RemarkName, UseYn) VALUES (%s, %s, %s)"

remark_values = [
    ('PRG', 'Progressing', 'Y'),
    ('BRD', 'Boarding', 'Y'),
    ('DPT', 'Departed', 'Y'),
    ('ARR', 'Arrived', 'Y'),
    ('DLT', 'Delayed', 'Y'),
    ('CNL', 'Cancelled', 'Y'),
    ('ONT', 'On Time', 'Y'),
    ('ERL', 'Early', 'Y')
]

cursor.executemany(sql, remark_values)
conn.commit()
conn.close()

## **Users, Roles, UserRoles**

In [247]:
conn = pymysql.connect(
    host='localhost',
    user='root',
    password='root',
    db='flight_data',
    port=8889,
    charset='utf8mb4'
)
cursor = conn.cursor()

sql_roles = """
INSERT INTO Roles (RoleID, RoleName) VALUES
('ADMIN', 'Administrator'),
('OPS', 'Operations Manager'),
('VIEWER', 'Viewer');
"""

sql_users = """
INSERT INTO Users (UserID, UserName, Password, AirportCode, AirlineCode) VALUES
('user01', 'Alice Johnson', '5e884898da28047151d0e56f8dc62927', 'ATL', 'AA'),
('user02', 'Bob Smith', '6cb75f652a9b52798eb6cf2201057c73', 'IND', 'DL'),
('user03', 'Charlie Kim', '2b3a8d0a34e3e237c54b4cf2d80a3c41', 'IND', 'AA');
"""

sql_user_roles = """
INSERT INTO UserRoles (UserRoleID, UserID, RoleID) VALUES
('UR01', 'user01', 'ADMIN'),
('UR02', 'user02', 'OPS'),
('UR03', 'user03', 'VIEWER');
"""

cursor.execute(sql_roles)
cursor.execute(sql_users)
cursor.execute(sql_user_roles)
conn.commit()
conn.close()

## **ActiveFlightSchedules**

In [248]:
from selenium import webdriver
from bs4 import BeautifulSoup
import time

driver = webdriver.Chrome()
driver.get("https://www.ind.com/flights/flight-status/arrivals?flightNumber=&airline=all&arrdep=A")

time.sleep(5) # Waiting unitl the data is loaded

soup = BeautifulSoup(driver.page_source, "html.parser")

theader = soup.select_one("thead.flight-table__header")

head = []
for row in theader.select("tr"):
    cols = row.find_all("th")
    head = [col.text.strip() for col in cols]

value = []
tbody = soup.select_one("tbody.flight-table__body")
for row in tbody.select("tr.flight-table__row"):
    cols = row.find_all("td")

    row_values = []

    for col in cols:
        text = col.text.strip()
        img = col.find("img")
        if img:
            img_src = 'https://www.ind.com'+img.get("src")
            combined = f"{text} ({img_src})" if text else img_src
        else:
            combined = text
        row_values.append(combined)

    value.append(row_values)

driver.quit()

In [249]:
import pandas as pd
df = pd.DataFrame(value, columns=head)
df

,Airline,Flight #,Origin,ETA,Gate #,Status
0,https://www.ind.com/assets/images/airlines/aa.png,AA 1255,Phoenix,1:54 am,B15,Arrived
1,https://www.ind.com/assets/images/airlines/wn.png,WN 4463,Baltimore,12:31 am,B23,Arrived
2,https://www.ind.com/assets/images/airlines/aa.png,AA 4520,Washington DC,1:11 am,HGR,Arrived
3,https://www.ind.com/assets/images/airlines/aa.png,AA 1033,Charlotte,12:03 am,B13,Arrived
4,https://www.ind.com/assets/images/airlines/ua.png,UA 6255,Wash. Dulles,12:00 am,A22,Arrived
...,...,...,...,...,...,...
138,https://www.ind.com/assets/images/airlines/aa.png,AA 2518,Miami,11:56 pm,,On Time
139,https://www.ind.com/assets/images/airlines/dl.png,DL 4909,Detroit,11:53 pm,A10,On Time
140,https://www.ind.com/assets/images/airlines/ua.png,UA 1956,Houston,11:57 pm,,On Time
141,https://www.ind.com/assets/images/airlines/aa.png,AA 766,Dallas-Ft. Worth,11:58 pm,,On Time


In [250]:
# Preprocessing
from datetime import datetime
import numpy as np

status_mapping = {
    'Progressing': 'PRG',
    'Boarding': 'BRD',
    'Departed': 'DPT',
    'Arrived': 'ARR',
    'Delayed': 'DLT',
    'Cancelled': 'CNL',
    'On Time': 'ONT',
    'Early': 'ERL'
}

df['FlightNumber'] = df['Flight #'].apply(lambda x: x.split(' ')[1])
df['AirlineCode'] = df['Flight #'].apply(lambda x: x.split(' ')[0])
df['AirportCode'] = 'IND' # Indianapolis
df['ScheduledDate'] = datetime.today().strftime('%Y%m%d')
df['ScheduledTime'] = df['ETA'].apply(
    lambda x: datetime.strptime(x.strip().lower(), "%I:%M %p").strftime("%H%M"))
df['EstimatedDate'] = datetime.today().strftime('%Y%m%d')
df['EstimatedTime'] = df['ETA'].apply(
    lambda x: datetime.strptime(x.strip().lower(), "%I:%M %p").strftime("%H%M"))
df['OriginDestAirport'] = np.random.choice(df_airports['AirportCode'].dropna().unique(), size=len(df)) # Arbitrary data as airports data is not perfectly matching with those of df

#df['CheckinCounter'] = 
df['BoardingGate'] = df['Gate #']
#df['BaggageClaimBelt'] = 
df['Remarks'] = df['Status'].map(status_mapping)

df

,Airline,Flight #,Origin,ETA,Gate #,Status,FlightNumber,AirlineCode,AirportCode,ScheduledDate,ScheduledTime,EstimatedDate,EstimatedTime,OriginDestAirport,BoardingGate,Remarks
0,https://www.ind.com/assets/images/airlines/aa.png,AA 1255,Phoenix,1:54 am,B15,Arrived,1255,AA,IND,20250415,0154,20250415,0154,PGD,B15,ARR
1,https://www.ind.com/assets/images/airlines/wn.png,WN 4463,Baltimore,12:31 am,B23,Arrived,4463,WN,IND,20250415,0031,20250415,0031,CUN,B23,ARR
2,https://www.ind.com/assets/images/airlines/aa.png,AA 4520,Washington DC,1:11 am,HGR,Arrived,4520,AA,IND,20250415,0111,20250415,0111,PDX,HGR,ARR
3,https://www.ind.com/assets/images/airlines/aa.png,AA 1033,Charlotte,12:03 am,B13,Arrived,1033,AA,IND,20250415,0003,20250415,0003,LAX,B13,ARR
4,https://www.ind.com/assets/images/airlines/ua.png,UA 6255,Wash. Dulles,12:00 am,A22,Arrived,6255,UA,IND,20250415,0000,20250415,0000,DUB,A22,ARR
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
138,https://www.ind.com/assets/images/airlines/aa.png,AA 2518,Miami,11:56 pm,,On Time,2518,AA,IND,20250415,2356,20250415,2356,DTW,,ONT
139,https://www.ind.com/assets/images/airlines/dl.png,DL 4909,Detroit,11:53 pm,A10,On Time,4909,DL,IND,20250415,2353,20250415,2353,MCI,A10,ONT
140,https://www.ind.com/assets/images/airlines/ua.png,UA 1956,Houston,11:57 pm,,On Time,1956,UA,IND,20250415,2357,20250415,2357,FLL,,ONT
141,https://www.ind.com/assets/images/airlines/aa.png,AA 766,Dallas-Ft. Worth,11:58 pm,,On Time,766,AA,IND,20250415,2358,20250415,2358,ECP,,ONT


In [ ]:
import pymysql

# 1. MySQL 연결
conn = pymysql.connect(
    host='localhost',
    user='root',
    password='root',
    db='flight_data',
    port=8889,
    charset='utf8mb4'
)
cursor = conn.cursor()
# df = pd.read_csv(f"flights_arrival_{df['AirportCode'][0]}.csv")

for _, row in df.iterrows():
    row = row.where(pd.notnull(row), None)
    sql = """
    INSERT INTO ActiveFlightSchedules (
        FlightNumber, AirportCode, AirlineCode,
        ScheduledDate, ScheduledTime,
        EstimatedDate, EstimatedTime,
        OriginDestAirport, Remarks
    ) VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s)
    """
    
    values = (
        row['FlightNumber'],
        row['AirportCode'],
        row['AirlineCode'],
        row['ScheduledDate'],
        row['ScheduledTime'],
        row['EstimatedDate'],
        row['EstimatedTime'],
        row['OriginDestAirport'],
        row['Remarks']
    )
    
    cursor.execute(sql, values)

conn.commit()
conn.close()